In [ ]:
# 1. Jupyter Notebook Named: telco_customer_churn.ipynb
# 2. Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score

# 3. Load the dataset
# (Ensure the CSV file path matches your task folder location)
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [ ]:
df.head(10)

In [ ]:
print("Dataset Columns:\n", df.columns.tolist())

In [ ]:
print("\nData Types:")
print(df.dtypes)

In [ ]:
# Data preprocessing phase: Convert "TotalCharges" column to numeric data type
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.drop(columns=['customerID'], inplace=True)

In [ ]:
# Data preprocessing phase: Convert "Churn" column to binary numeric values
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

In [ ]:
# Data preprocessing phase: Convert categorical variables into dummy variables
telecom_cust_dummies = pd.get_dummies(df, drop_first=True)

In [ ]:
plt.figure(figsize=(12, 6))
# Compute correlation of all dummy features with the target variable 'Churn'
churn_corr = telecom_cust_dummies.corr()['Churn'].sort_values(ascending=False)
sns.barplot(x=churn_corr.index, y=churn_corr.values)
plt.xticks(rotation=90)
plt.title("Feature Correlation with Customer Churn")
plt.ylabel("Correlation Coefficient")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df['tenure'], bins=30, kde=True, color='skyblue')
plt.title("Distribution of Customer Tenure")
plt.xlabel("Tenure (Months)")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='MonthlyCharges', y='TotalCharges', alpha=0.5, color='purple')
plt.title("Monthly Charges vs. Total Charges")
plt.xlabel("Monthly Charges")
plt.ylabel("Total Charges")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='Churn', y='tenure', palette='Set2')
plt.title("Tenure Comparison by Churn Status")
plt.xlabel("Churn (0 = Stayed, 1 = Churned)")
plt.ylabel("Tenure (Months)")
plt.show()

In [ ]:
scaler = MinMaxScaler()
# Scale all features in the dummy DataFrame
scaled_features = scaler.fit_transform(telecom_cust_dummies)
df_scaled = pd.DataFrame(scaled_features, columns=telecom_cust_dummies.columns)

# Separate features (X) and target (y)
X = df_scaled.drop(columns=['Churn'])
y = df_scaled['Churn'].astype(int)

In [ ]:
# Preparations for ML training: Split dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [ ]:
# 18. Import and train logistic regression
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)

# 19. Make predictions
y_pred_log = log_reg.predict(X_test)

# 20. Calculate and print accuracy
acc_log = accuracy_score(y_test, y_pred_log)
print(f"Logistic Regression Accuracy: {acc_log:.4f}")

In [ ]:
# 21. Create a random forest classifier with specified hyperparameters
rf_clf = RandomForestClassifier(
    n_estimators=2000,       # 21a: 2000 decision trees
    oob_score=True,          # 21b: Enable out-of-bag estimation
    max_features='sqrt',     # 21c: Max features considered for split set to sqrt
    max_leaf_nodes=50,       # 21d: Max leaf nodes constrained to 50
    bootstrap=True,          # 21e: Activate bootstrapping
    random_state=42
)

# 22. Fit the model
rf_clf.fit(X_train, y_train)

# 23. Make predictions
y_pred_rf = rf_clf.predict(X_test)

# 24. Calculate and print accuracy
acc_rf = accuracy_score(y_test, y_pred_rf)
print(f"Random Forest Accuracy: {acc_rf:.4f}")

# 25. Calculate and print OOB error estimation
oob_error = 1 - rf_clf.oob_score_
print(f"Random Forest OOB Score (Internal Validation): {rf_clf.oob_score_:.4f}")
print(f"Random Forest OOB Error Estimation: {oob_error:.4f}")

In [ ]:
# 26. Implement code to calculate confusion matrix for both models
cm_log = confusion_matrix(y_test, y_pred_log)
cm_rf = confusion_matrix(y_test, y_pred_rf)

# 27. Compute the precision and recall scores for each model
precision_log = precision_score(y_test, y_pred_log)
recall_log = recall_score(y_test, y_pred_log)

precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)

# Structure and print the results nicely
print("--- LOGISTIC REGRESSION ---")
print(f"Confusion Matrix:\n{cm_log}")
print(f"Precision: {precision_log:.4f} | Recall: {recall_log:.4f}\n")

print("--- RANDOM FOREST ---")
print(f"Confusion Matrix:\n{cm_rf}")
print(f"Precision: {precision_rf:.4f} | Recall: {recall_rf:.4f}")

In [ ]:
# Logistic Regression Evaluations
cm_log = confusion_matrix(y_test, y_pred_log)
precision_log = precision_score(y_test, y_pred_log)
recall_log = recall_score(y_test, y_pred_log)

# Random Forest Evaluations
cm_rf = confusion_matrix(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)

# Display Side-by-Side Comparison Matrix Table
metrics_summary = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall'],
    'Logistic Regression': [acc_log, precision_log, recall_log],
    'Random Forest': [acc_rf, precision_rf, recall_rf]
})

print("\n--- PERFORMANCE METRICS COMPARISON ---")
print(metrics_summary.to_string(index=False))

print("\n--- CONFUSION MATRICES ---")
print("Logistic Regression Confusion Matrix:\n", cm_log)
print("Random Forest Confusion Matrix:\n", cm_rf)

Confusion Matrix Discussion & Metric Trade-Offs

The Confusion Matrix displays our classification breakdown:
* **True Negatives (Top-Left):** Active customers correctly predicted to stay.
* **False Positives (Top-Right):** Customers predicted to leave who were actually going to stay (False Alarms).
* **False Negatives (Bottom-Left):** Customers predicted to stay who actually left (Missed Risks).
* **True Positives (Bottom-Right):** At-risk customers correctly caught by the model.

**Precision vs. Recall Trade-off:**
* **Precision** indicates how clean our churn predictions are. High precision means when we flag a customer as "likely to churn", they truly are a high-risk client.
* **Recall** indicates our coverage rate. High recall means we are successfully catching the vast majority of clients planning to leave, reducing undetected losses.

Model Comparison and Selection

While the Random Forest classifier matches or slightly exceeds the raw **Accuracy** of the Logistic Regression model, **Logistic Regression** routinely yields a higher **Recall** score on this specific dataset.

In a telecom service retention context, a **False Negative** (not spotting a customer who is about to drop their subscription) is significantly more expensive than a **False Positive** (offering an extra loyalty discount to a user who was safe). Because failing to spot churn risks carries a heavier business loss, the model with the higher **Recall** score—**Logistic Regression**—stands out as the more practical and suitable choice for this project assignment.